# nuReasoning → FiftyOne: the "decision-frame reveal" demo

> ## Prerequisites — read before you start
> **You must have FiftyOne 1.17 (or newer) installed in a Python virtual environment, and be
> running this notebook on that environment's kernel.** The reasoning panel in Step 8 is a FiftyOne
> plugin and relies on APIs introduced in 1.17. If you don't have it yet:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate     # Windows: .venv\Scripts\activate
> pip install "fiftyone>=1.17" imageio imageio-ffmpeg huggingface_hub
> python -m ipykernel install --user --name fiftyone-demo --display-name "Python (fiftyone-demo)"
> ```
>
> Then start Jupyter and select the **Python (fiftyone-demo)** kernel for this notebook. Step 1
> verifies the versions before anything else runs.

This notebook builds a **synchronized multi-view reasoning playback** where each long-tail clip
plays across its eight camera views plus a bird's-eye-view (BEV) map, with the nuReasoning
**Spatial / Decision / Counterfactual** annotations attached frame-by-frame — and the critical
**decision frame** tagged so you can scrub straight to it and reveal the counterfactual fork
("here's what the model did; here's the unsafe option it ruled out, and why").

**What you get at the end**
- One FiftyOne *grouped video dataset*: one group per clip, one slice per camera view + a `bev` slice.
- Frame-level `detections` on each camera (from Spatial reasoning).
- Frame-level `scene_description`, `decision_longitudinal`, `decision_lateral`, `reasoning_trace`
  (from Decision reasoning).
- A boolean `is_decision_frame` flag and a `counterfactual` text field at the critical frame.
- A saved view filtered to the decision frames, so the demo "pauses" exactly where it matters.
- A custom reasoning panel that shows the keyframe image and its decision/counterfactual.

**Design choices that follow the data schema**
- nuReasoning ships clips as directories with `metadata.json`, per-timestamp `ego_state/`,
  `annotations/`, `reasoning/`, plus eight camera images + LiDAR per frame at 10 Hz.
- FiftyOne's App plays **video**, not a stack of per-frame images, so Step 4 encodes each camera's
  image sequence into a small mp4. This is the same pattern FiftyOne's own LeRobot importer uses
  (group = episode/clip, camera views = slices, video frames carry the labels).
- The HD map is a `map.pkl`, which the App can't render natively, so we rasterize a BEV frame
  per timestamp with matplotlib and encode that into the `bev` slice.

> **Heads-up on data access.** This notebook is **real-data only**. The Hugging Face repo is
> gated/compliance-reviewed and ~1.5 TB, so you won't stream it live — Step 2.5 downloads only a
> few clip subtrees (a few hundred MB each), or uses clips you've already staged locally. You need
> accepted access to the dataset and a Hugging Face login before the download will work.


## Step 0 — Use the right kernel

Make sure this notebook is running on the kernel for the virtual environment where FiftyOne 1.17
is installed (see the prerequisites above).

- **JupyterLab / Notebook:** Kernel ▸ Change Kernel ▸ pick your FiftyOne environment's kernel.
- **VS Code:** click the kernel picker (top-right) and select the same environment.

The cell below prints the active interpreter so you can confirm it's the environment you expect.

In [ ]:
import sys
print("Python executable:", sys.executable)
print("If this isn't your FiftyOne virtual environment, switch kernels (Step 0) before continuing.")


## Step 1 — Verify FiftyOne 1.17 and supporting libraries

We rely on three capabilities that exist in FiftyOne 1.17: **grouped datasets**, **grouped *video*
slices**, and **frame-level custom fields**. We also need `ffmpeg` on the PATH (FiftyOne uses it to
probe/transcode video) and a couple of plain scientific-Python libs for BEV rasterization.

In [ ]:
import shutil, subprocess

import fiftyone as fo
print("FiftyOne version:", fo.__version__)

major, minor = (int(x) for x in fo.__version__.split(".")[:2])
assert (major, minor) >= (1, 17), (
    f"This notebook targets FiftyOne >= 1.17; found {fo.__version__}. "
    "Install/upgrade inside the venv: pip install 'fiftyone>=1.17,<1.18'"
)

# ffmpeg is required to encode the per-camera mp4s and for the App to play them back.
ffmpeg = shutil.which("ffmpeg")
print("ffmpeg:", ffmpeg or "NOT FOUND")
if ffmpeg:
    ver = subprocess.run([ffmpeg, "-version"], capture_output=True, text=True).stdout.splitlines()[0]
    print(" ", ver)
else:
    print("  Install ffmpeg (e.g. `sudo apt install ffmpeg` or `brew install ffmpeg`) before Step 4.")

# These are standard; they ship with most FiftyOne installs, but confirm.
import numpy as np
import matplotlib
print("numpy:", np.__version__, "| matplotlib:", matplotlib.__version__)
try:
    import imageio.v2 as imageio  # noqa: F401
    # The ffmpeg backend is what writes mp4s. Without it, imageio silently falls back to other
    # writers (e.g. TIFF) and the encode in Step 4 fails with a confusing `fps` TypeError.
    import imageio_ffmpeg
    print("imageio:", imageio.__version__, "| ffmpeg backend:", imageio_ffmpeg.get_ffmpeg_exe())
except Exception as _e:
    print("imageio ffmpeg backend missing ->", _e)
    print("   Fix: pip install imageio imageio-ffmpeg   (REQUIRED for Step 4 mp4 encoding)")

# Only needed if you fetch real data in Step 2.5; optional otherwise.
try:
    import huggingface_hub
    print("huggingface_hub:", huggingface_hub.__version__, "(needed only for the real-data fetch)")
except Exception:
    print("huggingface_hub not found -> `pip install huggingface_hub` (only needed for Step 2.5)")


## Step 2 — Configure paths and how many real clips to use

This notebook is **real-data only**. There is no synthetic fallback: it either downloads real
nuReasoning clips in Step 2.5 or uses clips you've already staged locally.

Edit the constants below.

- `DEMO_BUILD_DIR` is a writable scratch dir for the encoded mp4s and (if you let Step 2.5 fetch)
  the downloaded clips.
- `N_REAL_CLIPS` controls how many real clips the demo uses. Keep it small (2–4): each clip is
  ~201 frames × 8 cameras, so encoding and download both scale with this.
- `CLIP_DIRS` is optional. **Leave it empty** to let Step 2.5 discover and download clips from
  Hugging Face. **Or**, if you've already downloaded clips yourself, list the *clip directory*
  paths here (each must contain a `metadata.json`) and Step 2.5 will use them as-is instead of
  downloading. For the richest demo, pick clips spanning the four edge-case families from the
  Motional blog: VRU, aggressive vehicle behavior, environmental/scene conditions, and OOD objects.

In [ ]:
from pathlib import Path

# ---- EDIT ME -------------------------------------------------------------
DEMO_BUILD_DIR = Path.home() / "nureasoning_demo_build"   # writable scratch dir
N_REAL_CLIPS   = 3                                        # how many real clips to use (keep small)

# OPTION A (default): leave CLIP_DIRS empty -> Step 2.5 downloads real clips from Hugging Face.
# OPTION B: if you already downloaded clips, list their clip-directory paths here (each must
#           contain metadata.json); Step 2.5 will use these and skip downloading.
CLIP_DIRS = [
    # Path("/data/nuReasoning/data/train/part_1/<log>_<token>"),
]

# Camera slice names, in nuReasoning's ordering. The front camera carries the reasoning text.
CAMERA_SLICES = [
    "front", "front_left", "front_right",
    "left", "right",
    "back", "back_left", "back_right",
]
PRIMARY_CAMERA = "front"   # default slice shown in the App; where decision text lives
FPS = 10                   # nuReasoning sensors/state are sampled at 10 Hz
# --------------------------------------------------------------------------

NUREASONING_ROOT = None    # set by Step 2.5 (download root, or inferred from staged CLIP_DIRS)
CLIP_DIRS = [Path(p) for p in CLIP_DIRS]

DEMO_BUILD_DIR.mkdir(parents=True, exist_ok=True)
print("Build/scratch dir:", DEMO_BUILD_DIR)
print("Pre-staged clips :", len(CLIP_DIRS) if CLIP_DIRS else "none (will download in Step 2.5)")
print("Clips to use     :", N_REAL_CLIPS)


## Step 2.5 — Acquire the real clips

This step puts real nuReasoning clips on disk and populates `CLIP_DIRS`. Two modes, chosen
automatically:

- **You pre-staged clips** (set `CLIP_DIRS` in Step 2): it validates them and skips downloading.
- **You didn't**: it downloads `N_REAL_CLIPS` real clips from Hugging Face.

**Before the download path will work:**

1. The dataset is **gated**. Open https://huggingface.co/datasets/qixuewei/nuReasoning and
   request/accept access (approval is subject to Motional's compliance review).
2. Authenticate: run `huggingface-cli login` in a terminal *inside your FiftyOne virtual
   environment*, or set `HF_TOKEN` in your environment.
3. Mind the size. The full repo is ~1.5 TB; this cell downloads only the clip subtrees it selects
   (a few hundred MB each — eight camera image sequences + LiDAR + annotations per 20 s clip), so
   keep `N_REAL_CLIPS` small.

It discovers clip archives by scanning the repo for per-clip `*.zip` files, picks a few (spread
across `part_*` shards for scenario diversity), downloads just those, **unzips each into its own
clip directory**, and sets `NUREASONING_ROOT` / `CLIP_DIRS`. Step 3 onward is unchanged.

In [ ]:
import os, posixpath

REPO_ID      = "qixuewei/nuReasoning"
DOWNLOAD_DIR = DEMO_BUILD_DIR / "hf_real"   # where downloaded clips land

# ---- Mode B: pre-staged local clips -> validate and use them, no download. -----------------
staged = [c for c in CLIP_DIRS if (c / "metadata.json").exists()]
if CLIP_DIRS and len(staged) == len(CLIP_DIRS):
    NUREASONING_ROOT = None  # not needed; CLIP_DIRS are absolute
    print(f"Using {len(staged)} pre-staged clip(s); skipping download.")
    for c in staged:
        print("  ", c)
elif CLIP_DIRS and staged != CLIP_DIRS:
    missing = [str(c) for c in CLIP_DIRS if c not in staged]
    raise RuntimeError(
        "Some CLIP_DIRS have no metadata.json:\n  " + "\n  ".join(missing) +
        "\nFix the paths, or clear CLIP_DIRS in Step 2 to download instead."
    )
else:
    # ---- Mode A: download a real subset from Hugging Face. ----------------------------------
    # NOTE: nuReasoning stores each clip as a single `*.zip` archive under data/<split>/<part>/.
    #       metadata.json lives INSIDE each zip, so we discover archives, download a few, and
    #       unzip each into its own clip directory.
    import zipfile
    try:
        from huggingface_hub import HfApi, hf_hub_download
        from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
    except Exception as e:
        raise RuntimeError(
            "huggingface_hub is required to download. Install it in the venv: "
            "pip install huggingface_hub"
        ) from e

    # Auth: use HF_TOKEN if set, else any cached `huggingface-cli login` (token=True).
    token = os.environ.get("HF_TOKEN") or True
    api = HfApi()

    # Discover per-clip zip archives in the repo listing.
    try:
        all_files = api.list_repo_files(REPO_ID, repo_type="dataset", token=token)
    except GatedRepoError as e:
        raise RuntimeError(
            "Access to qixuewei/nuReasoning is gated. Accept access on the dataset page, then "
            "`huggingface-cli login` (or set HF_TOKEN) and re-run this cell."
        ) from e
    except HfHubHTTPError as e:
        raise RuntimeError(f"Could not list repo files: {e}. Check auth/network and retry.") from e

    zip_files = sorted(f for f in all_files if f.lower().endswith(".zip") and f.startswith("data/"))
    if not zip_files:
        raise RuntimeError(
            "No per-clip *.zip archives found under data/ in the repo listing. "
            f"Sample of files seen: {all_files[:10]}"
        )

    # Prefer the train split; spread picks across distinct part_* shards for scenario diversity.
    pool = [z for z in zip_files if "/train/" in z] or zip_files
    picked, seen_parts = [], set()
    for z in pool:
        parts = z.split("/")
        part = parts[2] if len(parts) > 2 else z  # data/<split>/<part>/<clip>.zip
        if part not in seen_parts:
            picked.append(z); seen_parts.add(part)
        if len(picked) >= N_REAL_CLIPS:
            break
    for z in pool:                      # top up if there weren't enough distinct shards
        if len(picked) >= N_REAL_CLIPS:
            break
        if z not in picked:
            picked.append(z)

    print(f"Selected {len(picked)} clip archive(s) to download:")
    for z in picked:
        print("  ", z)

    # Download + unzip each archive into data_unzipped/<split>/<part>/<clip_name>/,
    # mirroring the dataset card's recommended extraction layout.
    NUREASONING_ROOT = DOWNLOAD_DIR / "data_unzipped"
    CLIP_DIRS = []
    for z in picked:
        local_zip = hf_hub_download(
            repo_id=REPO_ID, repo_type="dataset", token=token,
            filename=z, local_dir=str(DOWNLOAD_DIR),
        )
        rel = z[len("data/"):] if z.startswith("data/") else z   # <split>/<part>/<clip>.zip
        clip_name = posixpath.basename(rel)[:-4]                 # strip ".zip"
        out_dir = NUREASONING_ROOT / posixpath.dirname(rel) / clip_name
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(local_zip) as zf:
            zf.extractall(out_dir)
        # Layout-agnostic: the archive's internal nesting varies, so locate metadata.json anywhere
        # under the extraction root and use its parent directory as the clip dir.
        found = None
        for root, _dirs, files in os.walk(out_dir):
            if "metadata.json" in files:
                found = Path(root)
                break
        if found is None:
            print(f"[warn] no metadata.json anywhere under {out_dir}; "
                  f"top-level contents: {[p.name for p in out_dir.iterdir()]}")
        else:
            CLIP_DIRS.append(found)

    usable = [c for c in CLIP_DIRS if (c / "metadata.json").exists()]
    print()
    print("Unzipped under:", NUREASONING_ROOT)
    print(f"Usable clips  : {len(usable)} / {len(picked)} archive(s)")
    for c in usable:
        print("  ", c)
    assert usable, (
        "Downloaded/unzipped but no metadata.json found under any archive. "
        "Inspect the [warn] lines above to see what the archives actually contain."
    )


## Step 3 — Loader: walk a clip directory into a plain Python structure

We keep the FiftyOne specifics out of this step. The goal here is a clean intermediate
representation per clip:

```
clip = {
    "clip_token", "scenario_type", "location",
    "frames": [
        {
            "frame_index", "timestamp_us", "relative_time_s",
            "cameras": {slice_name: image_path, ...},
            "ego": {...},                      # from ego_state/<ts>.pkl
            "annotations": {...},              # from annotations/<ts>.pkl
            "reasoning": {Spatial, Driving, Counterfactual},  # from reasoning/<ts>.json
        }, ...
    ],
}
```

The loader reads defensively: nuReasoning's per-frame `annotations`/`reasoning` schemas vary, so
missing or differently-named keys degrade to empty fields rather than raising. **Reasoning is
sparse** — only labeled keyframes (roughly one every ~10 frames) carry it — so the loader marks
each frame with `is_keyframe` and **forward-fills** the most recent reasoning across the gaps, so
playback shows the current annotation continuously instead of flickering empty.

In [ ]:
import json, pickle

# The ego_state/ and annotations/ pickles were saved with nuReasoning's own devkit classes
# (e.g. a `data_schema` module that isn't on PyPI). Unpickling them requires that module on the
# import path, which we don't have. These pickles are NOT needed for the demo: the scene text,
# decisions, and counterfactuals all come from the plain-JSON reasoning files. So we load pickles
# best-effort and return None on any failure (missing module, custom class, corrupt file),
# warning once so it's visible but non-fatal.
_PICKLE_WARNED = set()

def _load_pickle(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except (ModuleNotFoundError, AttributeError, ImportError, pickle.UnpicklingError, EOFError) as e:
        kind = type(e).__name__
        if kind not in _PICKLE_WARNED:
            print(f"[note] skipping pickle payloads ({kind}: {e}). "
                  f"These feed only optional 2D boxes / BEV markers; reasoning JSON is unaffected.")
            _PICKLE_WARNED.add(kind)
        return None

# ---- Reasoning-JSON coercion helpers --------------------------------------------------------
# The real reasoning/*.json shapes vary from the dataset-card example (e.g. "Driving decision"
# may be a plain string OR a {"Longitudinal","Lateral"} dict). These helpers read either shape
# without raising, so downstream code never assumes a specific structure.
def _as_dict(v):
    return v if isinstance(v, dict) else {}

def _as_list(v):
    if isinstance(v, list):
        return v
    return [] if v in (None, "", {}) else [v]

def _as_text(v):
    if v is None:
        return ""
    if isinstance(v, str):
        return v
    if isinstance(v, dict):
        parts = [str(v[k]) for k in ("Longitudinal", "Lateral") if k in v and v[k]]
        if not parts:
            parts = [str(x) for x in v.values() if x not in (None, "")]
        return "; ".join(parts)
    return str(v)

def _decision_pair(drv):
    '''Return (longitudinal, lateral) from a Driving block, tolerating dict or string forms.'''
    dec = _as_dict(drv).get("Driving decision")
    if isinstance(dec, dict):
        return _as_text(dec.get("Longitudinal", "")), _as_text(dec.get("Lateral", ""))
    return _as_text(dec), ""

def load_real_clip(clip_dir: Path) -> dict:
    '''Parse one nuReasoning clip directory into the intermediate structure.'''
    meta = json.loads((clip_dir / "metadata.json").read_text())
    frames_out = []
    for fr in meta.get("frames", []):
        ts = fr["timestamp_us"]
        cams = {}
        for slc in CAMERA_SLICES:
            rel = fr.get("sensors", {}).get("cameras", {}).get(slc)
            if rel:
                cams[slc] = str((clip_dir / rel).resolve())

        ego = annotations = None
        ego_rel = fr.get("ego_state")
        if ego_rel and (clip_dir / ego_rel).exists():
            ego = _load_pickle(clip_dir / ego_rel)
        ann_rel = fr.get("annotations")
        if ann_rel and (clip_dir / ann_rel).exists():
            annotations = _load_pickle(clip_dir / ann_rel)

        reasoning = {}
        rsn_rel = fr.get("reasoning")
        if rsn_rel and (clip_dir / rsn_rel).exists():
            reasoning = json.loads((clip_dir / rsn_rel).read_text())

        frames_out.append({
            "frame_index": fr["frame_index"],
            "timestamp_us": ts,
            "relative_time_s": fr.get("relative_time_s", fr["frame_index"] / FPS),
            "cameras": cams,
            "ego": ego,
            "annotations": annotations,
            "reasoning": reasoning,
            "is_keyframe": bool(reasoning),     # reasoning is sparse: only keyframes carry it
            "mission_goal": fr.get("mission_goal", {}),
        })

    # Camera 2D boxes in the reasoning are ABSOLUTE PIXELS, so we need each camera's image size to
    # normalize them for FiftyOne. Prefer the exact width/height from metadata camera_calibrations
    # (keyed by nuScenes-style CAM_M_* names); fall back to reading one image per camera via PIL.
    CAM_KEY_TO_SLICE = {
        "CAM_M_F": "front", "CAM_M_F0": "front",
        "CAM_M_L0": "front_left", "CAM_M_L1": "left", "CAM_M_L2": "back_left",
        "CAM_M_R0": "front_right", "CAM_M_R1": "right", "CAM_M_R2": "back_right",
        "CAM_M_B": "back",
    }
    cam_size = {}   # slice -> (width, height)
    for cam_key, cal in (meta.get("camera_calibrations") or {}).items():
        slc = CAM_KEY_TO_SLICE.get(cam_key)
        if slc and isinstance(cal, dict) and cal.get("width") and cal.get("height"):
            cam_size[slc] = (int(cal["width"]), int(cal["height"]))

    if len(cam_size) < len(CAMERA_SLICES):   # fill any gaps by reading an actual image
        from PIL import Image
        for fr in frames_out:
            for slc, p in fr["cameras"].items():
                if slc not in cam_size and os.path.exists(p):
                    try:
                        with Image.open(p) as im:
                            cam_size[slc] = im.size  # (w, h)
                    except Exception:
                        pass
            if len(cam_size) == len(CAMERA_SLICES):
                break

    # Forward-fill reasoning across the gaps between keyframes so every frame shows the current
    # annotation during playback instead of flickering empty. IMPORTANT: the sub-blocks update at
    # different rates (Spatial on ~13 keyframes, Driving/Counterfactual on ~3), so we forward-fill
    # each sub-block INDEPENDENTLY — otherwise a stale Spatial block would get paired with a fresh
    # Driving block (or vice versa). is_keyframe still marks frames that originally carried labels.
    SUBKEYS = ("Spatial", "Driving", "Counterfactual",
               "cross_view_correspondence", "object_relations", "map")
    def _is_filled(v):
        return v not in (None, "", {}, [])
    last = {}
    for fr in frames_out:
        src = fr["reasoning"] if isinstance(fr["reasoning"], dict) else {}
        for k in SUBKEYS:
            if _is_filled(src.get(k)):
                last[k] = src.get(k)
        # rebuild this frame's reasoning from the most recent filled sub-blocks
        fr["reasoning"] = dict(last)

    return {
        "clip_token": meta.get("clip_token", clip_dir.name),
        "scenario_type": meta.get("scenario_type", "unknown"),
        "location": meta.get("clip_location", "unknown"),
        "cam_size": cam_size,
        "map_pkl": str((clip_dir / meta["map_annotation"]).resolve())
                   if meta.get("map_annotation") and (clip_dir / meta["map_annotation"]).exists()
                   else None,
        "frames": frames_out,
    }

print("Real-clip loader defined.")


In [ ]:
# Build the working list of clips from the real clip directories acquired in Step 2.5.
clips = []
for cd in CLIP_DIRS:
    if not (cd / "metadata.json").exists():
        print(f"[skip] no metadata.json in {cd}")
        continue
    clips.append(load_real_clip(cd))

assert clips, (
    "No clips loaded. Run Step 2.5 to download real clips, or set CLIP_DIRS in Step 2 to "
    "valid local clip directories (each containing metadata.json)."
)
for c in clips:
    print(f"{c['clip_token']:42s} | {c['scenario_type']:24s} | {len(c['frames'])} frames | {c['location']}")


## Step 4 — Encode each camera view (and a BEV) into mp4 clips

FiftyOne plays video samples, so we turn each camera's ordered image sequence into one short mp4.
We also rasterize a **bird's-eye-view** per frame and encode that as an extra `bev` slice — that's
the panel that makes the spatial story legible to a non-technical audience.

- Real data: if a `map.pkl` is present we draw lanes/crosswalks/ego pose from it; otherwise we fall
  back to plotting the ego trajectory + nearby objects from `annotations`.
- We re-encode to **H.264 / yuv420p** so the clips play in the browser-based App (same requirement
  the FiftyOne LeRobot importer documents).
- We **downscale camera frames to 960px wide** on encode. nuReasoning cameras are full sensor
  resolution (~2816×1856); decoding eight of those plus the BEV simultaneously in a grouped view
  pins the App on its "Pixelating…" loading state. 960px is plenty for a demo and makes the grid
  paint instantly. Detections are stored normalized to [0,1], so downscaling doesn't shift them.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio

VIDEO_DIR = DEMO_BUILD_DIR / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

def _encode_images_to_mp4(image_paths, out_path: Path, fps=FPS, target_w=960):
    '''Browser-friendly H.264/yuv420p mp4. Downscales wide frames to target_w (keeping aspect,
    even dimensions) so the App can decode 8 cameras + BEV in a grouped view without stalling.
    Full sensor res (e.g. 2816x1856) x 9 slices is what pins the App on "pixelating" forever.
    '''
    from PIL import Image
    import numpy as _np
    # Force the FFMPEG backend explicitly. If imageio can't resolve it (missing imageio-ffmpeg),
    # it otherwise silently falls back to e.g. the TIFF writer, which then errors on `fps`.
    writer = imageio.get_writer(
        out_path, format="FFMPEG", mode="I", fps=fps, codec="libx264",
        macro_block_size=2,        # pad odd dims to a multiple of 2; imageio-ffmpeg then emits yuv420p
        # NB: no explicit -pix_fmt here -- the writer already sets yuv420p, and passing it again
        # triggers ffmpeg's "Multiple -pix_fmt options specified" warning on every stream.
    )
    try:
        for p in image_paths:
            im = imageio.imread(p)
            if im.ndim == 3 and im.shape[2] == 4:   # drop alpha (RGBA -> RGB)
                im = im[:, :, :3]
            h, w = im.shape[:2]
            if target_w and w > target_w:           # downscale only; never upscale
                new_w = target_w
                new_h = max(2, int(round(h * target_w / w)))
                new_h -= new_h % 2                   # force even height (yuv420p needs even dims)
                im = _np.asarray(
                    Image.fromarray(im).resize((new_w, new_h), Image.BILINEAR)
                )
            writer.append_data(im)
    finally:
        writer.close()
    return out_path

def _render_bev_frame(clip, frame, out_path: Path):
    '''One BEV PNG for a frame: ego at origin, heading up, objects/trajectory around it.'''
    fig, ax = plt.subplots(figsize=(4, 4), dpi=110)
    ax.set_facecolor("#101218")
    ax.set_xlim(-30, 30); ax.set_ylim(-15, 45)
    ax.set_aspect("equal"); ax.axis("off")

    # lane hint
    for lx in (-3.7, 0.0, 3.7):
        ax.plot([lx, lx], [-15, 45], color="#3a4150", lw=1, ls=(0, (6, 6)))

    # ego vehicle
    ax.add_patch(plt.Rectangle((-1.0, -2.3), 2.0, 4.6, color="#4da3ff", zorder=5))
    ax.text(0, 6, "ego", color="#9cc4ff", ha="center", fontsize=8)

    # Objects in the ego frame, from Spatial.per_camera_results[*].objects[].detection_bbox_3d.
    # nuReasoning ego convention: center_3d_ego.x = forward (ahead+), y = left (+). Our BEV has the
    # y-axis pointing forward and x pointing right, so we plot at (right, forward) = (-y_ego, x_ego).
    # Objects are seen from multiple cameras, so dedupe by track_token.
    spatial = _as_dict((frame.get("reasoning") or {}).get("Spatial"))
    per_cam = _as_dict(spatial.get("per_camera_results") or spatial.get("Per_camera_results"))

    seen_tracks = set()
    plotted = 0
    for cam_name, cam in per_cam.items():
        for o in _as_list(_as_dict(cam).get("objects")):
            o = _as_dict(o)
            tok = o.get("track_token")
            if tok and tok in seen_tracks:
                continue
            c3d = _as_dict(_as_dict(o.get("detection_bbox_3d")).get("center_3d_ego"))
            if "x" not in c3d or "y" not in c3d:
                continue
            try:
                fwd = float(c3d["x"])      # ahead of ego
                left = float(c3d["y"])     # left of ego
            except (TypeError, ValueError):
                continue
            bx, by = -left, fwd            # BEV: x = right(+), y = forward(+)
            if tok:
                seen_tracks.add(tok)
            ax.add_patch(plt.Rectangle((bx - 1.0, by - 2.0), 2.0, 4.0, color="#ff5a5a", zorder=5))
            lbl = o.get("detection_label") or o.get("category") or "obj"
            ax.text(bx + 1.5, by, str(lbl), color="#ffb0b0", fontsize=6)
            plotted += 1
            if plotted >= 20:
                break
        if plotted >= 20:
            break
    if plotted == 0:
        ax.text(0, 40, "(no ego-frame object positions in this frame)",
                color="#5b6473", ha="center", fontsize=6)

    drv = _as_dict((frame.get("reasoning") or {}).get("Driving"))
    lon, _lat = _decision_pair(drv)
    ax.set_title(f"BEV  t={frame['relative_time_s']:.1f}s   {lon}", color="#dde3ee", fontsize=8)

    # Fixed, even-dimensioned canvas: 4in x 4in @ 110dpi = 440x440 (both even).
    # Do NOT use bbox_inches="tight" -- it yields odd sizes that libx264/yuv420p rejects.
    fig.savefig(out_path, facecolor=fig.get_facecolor())
    plt.close(fig)

def build_clip_videos(clip) -> dict:
    '''Returns {slice_name: mp4_path} for all cameras + 'bev' for one clip.'''
    token = clip["clip_token"]
    out = {}

    # cameras
    for slc in CAMERA_SLICES:
        imgs = [fr["cameras"].get(slc) for fr in clip["frames"] if fr["cameras"].get(slc)]
        if len(imgs) < 2:
            continue
        mp4 = VIDEO_DIR / f"{token}__{slc}.mp4"
        _encode_images_to_mp4(imgs, mp4)
        out[slc] = str(mp4)

    # bev
    bev_png_dir = DEMO_BUILD_DIR / "bev_src" / token
    bev_png_dir.mkdir(parents=True, exist_ok=True)
    bev_imgs = []
    for fr in clip["frames"]:
        p = bev_png_dir / f"{fr['frame_index']:04d}.png"
        _render_bev_frame(clip, fr, p)
        bev_imgs.append(str(p))
    bev_mp4 = VIDEO_DIR / f"{token}__bev.mp4"
    _encode_images_to_mp4(bev_imgs, bev_mp4)
    out["bev"] = str(bev_mp4)
    return out

# Encode everything (small clips -> fast).
clip_videos = {}
for c in clips:
    clip_videos[c["clip_token"]] = build_clip_videos(c)
    made = ", ".join(sorted(clip_videos[c["clip_token"]]))
    print(f"{c['clip_token']:42s} -> {made}")


## Step 5 — Build the grouped video dataset and attach frame-level reasoning

Now the FiftyOne part. For each clip we create one `fo.Group`. For each camera (and `bev`) we add a
**video sample** in that group's slice. Then we populate **frame-level** fields:

- On **every** camera slice: `detections` (from Spatial per-camera 2D boxes when present).
- On the **front** slice only (to avoid eight copies of the same text): `scene_description`,
  `decision_longitudinal`, `decision_lateral`, `reasoning_trace`, `is_decision_frame`,
  `counterfactual`.

FiftyOne frame indices are **1-based**, so we map nuReasoning `frame_index` (0-based) to `+1`.

In [ ]:
import fiftyone as fo

DATASET_NAME = "nureasoning_demo"
if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME, persistent=True)

ALL_SLICES = CAMERA_SLICES + ["bev"]

def _spatial_detections_for(slice_name, frame, cam_size):
    '''2D detections for a camera slice from Spatial.per_camera_results.

    Real nuReasoning schema: reasoning["Spatial"]["per_camera_results"][slice]["objects"] is a
    list of objects, each with "detection_bbox_2d" = [x1, y1, x2, y2] in ABSOLUTE PIXELS and a
    "detection_label"/"category". FiftyOne wants [x, y, w, h] normalized to [0, 1], so we divide
    by the camera image size. Falls back gracefully if keys/sizes are missing.
    '''
    spatial = _as_dict((frame.get("reasoning") or {}).get("Spatial"))
    per_cam = _as_dict(spatial.get("per_camera_results") or spatial.get("Per_camera_results"))
    cam = _as_dict(per_cam.get(slice_name))
    objs = _as_list(cam.get("objects"))
    if not objs:
        return None

    wh = cam_size.get(slice_name)
    dets = []
    for o in objs:
        o = _as_dict(o)
        bb = o.get("detection_bbox_2d")
        label = o.get("detection_label") or o.get("category") or "object"
        if not (isinstance(bb, (list, tuple)) and len(bb) == 4):
            continue
        x1, y1, x2, y2 = (float(v) for v in bb)
        if wh:
            W, H = wh
            nx, ny = x1 / W, y1 / H
            nw, nh = (x2 - x1) / W, (y2 - y1) / H
        elif max(bb) <= 1.0:           # already normalized
            nx, ny, nw, nh = x1, y1, x2 - x1, y2 - y1
        else:
            continue                    # abs pixels but unknown size -> can't normalize, skip
        # clamp to [0,1] so FiftyOne accepts it even if a box runs off-frame
        nx, ny = max(0.0, nx), max(0.0, ny)
        nw, nh = max(0.0, min(nw, 1.0 - nx)), max(0.0, min(nh, 1.0 - ny))
        if nw <= 0 or nh <= 0:
            continue
        dets.append(fo.Detection(label=str(label), bounding_box=[nx, ny, nw, nh]))
    return fo.Detections(detections=dets) if dets else None

def _fmt_action(a):
    '''Format one counterfactual action dict from the real schema.

    Real keys: {"Longitudinal", "Lateral", "Risk level", "Reason"} (older/example data may use
    {"action","risk","outcome"}). Returns a single "[Risk] longitudinal + lateral — reason" line.
    '''
    a = _as_dict(a)
    risk = a.get("Risk level") or a.get("risk") or a.get("Risk") or "?"
    lon = a.get("Longitudinal") or a.get("action") or a.get("Action") or ""
    lat = a.get("Lateral") or ""
    move = ", ".join(p for p in (lon, lat) if p and p.lower() not in ("no lateral action",)) or _as_text(a.get("action"))
    reason = a.get("Reason") or a.get("outcome") or a.get("Outcome") or a.get("explanation") or ""
    line = f"[{risk}] {move}"
    if reason:
        line += f" — {reason}"
    return line.strip(" —")

def _format_counterfactual(cf) -> str:
    cf = _as_dict(cf)
    out = []
    alts = _as_list(cf.get("Alternative actions"))
    crit = _as_list(cf.get("Top safety-critical actions"))
    if alts:
        out.append("Alternatives considered:")
        out.extend("  " + _fmt_action(a) for a in alts)
    if crit:
        out.append("Safety-critical actions to avoid:")
        out.extend("  " + _fmt_action(a) for a in crit)
    return "\n".join(l for l in out if l.strip())

for clip in clips:
    token = clip["clip_token"]
    vids = clip_videos[token]
    group = fo.Group()
    samples = []

    for slc in ALL_SLICES:
        if slc not in vids:
            continue
        s = fo.Sample(filepath=vids[slc], group=group.element(slc))
        # clip-level metadata duplicated on each slice for easy filtering in the App
        s["clip_token"] = token
        s["scenario_type"] = clip["scenario_type"]
        s["location"] = clip["location"]

        # frame-level fields
        cam_size = clip.get("cam_size", {})
        for fr in clip["frames"]:
            fno = fr["frame_index"] + 1   # FiftyOne frames are 1-based
            rsn = _as_dict(fr.get("reasoning"))
            drv = _as_dict(rsn.get("Driving"))

            # detections on camera slices (not bev)
            if slc in CAMERA_SLICES:
                d = _spatial_detections_for(slc, fr, cam_size)
                if d is not None:
                    s.frames[fno]["detections"] = d

            # reasoning text only on the primary camera to avoid duplication
            if slc == PRIMARY_CAMERA:
                cf = _as_dict(rsn.get("Counterfactual"))
                lon, lat = _decision_pair(drv)
                has_cf = bool(_as_list(cf.get("Alternative actions")) or
                              _as_list(cf.get("Top safety-critical actions")))
                # "Critical components" is a dict of actors/signals -> attributes; render compactly
                crit = _as_dict(drv.get("Critical components"))
                crit_txt = "\n".join(
                    f"{name}: " + "; ".join(f"{k}={v}" for k, v in _as_dict(attrs).items())
                    for name, attrs in crit.items()
                ) if crit else ""
                s.frames[fno]["scene_description"]     = _as_text(drv.get("Scene description"))
                s.frames[fno]["critical_components"]   = crit_txt
                s.frames[fno]["decision_longitudinal"] = lon
                s.frames[fno]["decision_lateral"]      = lat
                s.frames[fno]["reasoning_trace"]       = _as_text(drv.get("Reasoning trace"))
                s.frames[fno]["is_keyframe"]           = bool(fr.get("is_keyframe"))
                s.frames[fno]["is_decision_frame"]     = has_cf
                s.frames[fno]["counterfactual"]        = _format_counterfactual(cf)

        # clip-level flags (sample scope) so the App can match without frame-flattening
        if slc == PRIMARY_CAMERA:
            s["has_keyframe"] = any(fr.get("is_keyframe") for fr in clip["frames"])
            s["has_decision_frame"] = any(
                bool(_as_list(_as_dict(_as_dict(fr.get("reasoning")).get("Counterfactual")).get("Alternative actions"))
                     or _as_list(_as_dict(_as_dict(fr.get("reasoning")).get("Counterfactual")).get("Top safety-critical actions")))
                for fr in clip["frames"]
            )

        samples.append(s)

    dataset.add_samples(samples)

dataset.default_group_slice = PRIMARY_CAMERA
print("Group slices:", dataset.group_slices)
print("Default slice:", dataset.default_group_slice)
print("Num groups (clips):", len(dataset.distinct("group.id")))
print("Media type:", dataset.media_type)


## Step 6 — Tag scenario type on the sidebar and label the decision frames

Two small touches make the live demo smoother:

1. Promote `scenario_type`, `clip_token`, and `location` so they're obvious in the App sidebar.
2. Build a **saved view** restricted to the decision frames, so during the demo you click one
   thing and land on the exact moment the counterfactual matters.

In [ ]:
from fiftyone import ViewField as F

# 1) Make the clip-level fields easy to find / filter in the App.
dataset.app_config.sidebar_groups = fo.DatasetAppConfig.default_sidebar_groups(dataset)

# 1b) Disable frame filtering. This is the key stability fix for THIS dataset: the slices have an
#     asymmetric frame schema (camera slices carry `detections`, the front slice carries the
#     reasoning text fields, and the `bev` slice has no frame fields at all). When the App builds
#     its per-frame filter machinery over that uneven schema it can throw a backend error that
#     surfaces in the browser as "Cannot read properties of undefined (reading 'stack')".
#     Turning off frame filtering avoids that path entirely; per-frame values still display fine.
dataset.app_config.disable_frame_filtering = True
dataset.save()

# 2) Saved views (kept GROUPED — do NOT flatten with select_group_slices(...).match_frames(...),
#    which can crash the App's video grid). Matching on clip-level booleans keeps media type intact.
#    Reasoning is sparse (only ~13 keyframes per 202-frame clip), so we expose two entry points:
#      - "keyframes": clips that carry any labeled keyframe (the navigable, informative moments)
#      - "decision_frames": the subset that also has counterfactual alternatives (the reveal)
keyframe_view = dataset.match(F("has_keyframe") == True)
decision_view = dataset.match(F("has_decision_frame") == True)
print("Clips with a keyframe   :", len(keyframe_view))
print("Clips with a decision   :", len(decision_view))

for name, view in [("keyframes", keyframe_view), ("decision_frames", decision_view)]:
    if name in dataset.list_saved_views():
        dataset.delete_saved_view(name)
    dataset.save_view(name, view)
SAVED_VIEW = "decision_frames"
print("Saved views:", dataset.list_saved_views())


## Step 7 — Launch the App and run the demo

`fo.launch_app(dataset)` opens the App. In a notebook it renders inline; you can also open it in a
browser tab. The flow you'll narrate:

1. **Grid view** shows the front camera for each clip. Point out the `scenario_type` sidebar field —
   that's the long-tail taxonomy (VRU / vehicle behavior / environment / OOD object).
2. **Open a clip** (expand the modal). The left carousel lets you switch slices — flip through the
   eight cameras and the `bev` slice to show the synchronized multi-view context.
3. **Scrub the timeline.** As you move frame-to-frame, the sidebar shows `scene_description`,
   `decision_longitudinal/lateral`, and `reasoning_trace` updating per frame.
4. **The reveal.** Switch to the saved **`decision_frames`** view (saved-views menu) to jump
   straight to the critical frame, then read out the `counterfactual` field: what the model did vs.
   the unsafe/suboptimal alternatives it ruled out and why. That single beat is the demo.

In [ ]:
# Open the App on the grouped dataset, default (front) slice, no active view.
dataset.group_slice = PRIMARY_CAMERA

session = fo.launch_app(dataset)
session.view = None
session.refresh()
session


In [ ]:
# To jump the App straight to the decision-frame view during the demo:
session.view = dataset.load_saved_view("decision_frames")

# To bring everything back:
# session.view = None


## Step 8 — Build the synced reasoning panel (plugin)

The sidebar already shows per-frame reasoning, but for the demo we build a dedicated,
presentation-sized **"nuReasoning" panel** that sits next to the video in the expanded modal and
shows the current clip's labeled keyframes — scene description, critical components, the driving
decision, and the counterfactual fork — in large, readable type.

A FiftyOne panel is a real plugin: a directory containing `fiftyone.yml` + `__init__.py`,
discovered via the plugins directory. This step is **part of the build**, not optional. We:

1. Stage each clip's keyframe annotations in the dataset's **execution store** (a key-value store
   keyed by `clip_token`). We deliberately do NOT add a big string field to the samples — a large
   field present on only the front-camera slice creates a per-slice schema asymmetry that crashes
   the App's dataset load (the opaque "Cannot read properties of undefined (reading 'stack')"
   error). The execution store lives outside the sample schema, so it avoids that entirely.
2. Write the plugin files to a local plugins directory and point FiftyOne at it.
3. Register the panel and open it in the modal beside the playback.

The panel updates on `on_change_current_sample` (when you open/switch clips) and offers a keyframe
selector so you can jump straight to the decision keyframe and read out the counterfactual.

In [ ]:
import json as _json

# 1) Stage each clip's keyframe reasoning for the panel.
#    IMPORTANT: we do NOT add a big string field to the samples. A large field present on only the
#    front-camera slice creates a per-slice schema asymmetry that can crash the App's dataset load
#    (the opaque "Cannot read properties of undefined (reading 'stack')" error). Instead we keep
#    the panel's data in the dataset's EXECUTION STORE (a key-value store that is NOT part of the
#    sample schema), keyed by clip_token. The panel reads it back the same way.
def _clip_keyframe_summary(clip):
    out = []
    for fr in clip["frames"]:
        if not fr.get("is_keyframe"):
            continue
        rsn = _as_dict(fr.get("reasoning"))
        drv = _as_dict(rsn.get("Driving"))
        cf = _as_dict(rsn.get("Counterfactual"))
        lon, lat = _decision_pair(drv)
        crit = _as_dict(drv.get("Critical components"))
        out.append({
            "frame": fr["frame_index"] + 1,                  # 1-based, matches the video timeline
            "t": round(fr["relative_time_s"], 1),
            "scene": _as_text(drv.get("Scene description")),
            "critical": {k: _as_dict(v) for k, v in crit.items()},
            "decision": {"Longitudinal": lon, "Lateral": lat},
            "trace": _as_text(drv.get("Reasoning trace")),
            "counterfactual": _format_counterfactual(cf),
            "image": fr.get("cameras", {}).get(PRIMARY_CAMERA),  # front-camera frame for the panel
        })
    return out

token_to_summary = {c["clip_token"]: _clip_keyframe_summary(c) for c in clips}

# Make sure no stale per-sample field lingers from an earlier run (keeps the sample schema clean).
if "reasoning_summary" in dataset.get_field_schema():
    dataset.delete_sample_field("reasoning_summary")
    dataset.save()

# Write the summaries to the dataset execution store under the "nureasoning_panel" namespace.
# Use the version-stable ExecutionStore API (dataset.create_store(...) is a newer convenience
# wrapper that some 1.17 builds don't expose). Scope the store to this dataset via dataset_id.
try:
    store = dataset.create_store("nureasoning_panel")          # newer convenience method
except AttributeError:
    from fiftyone.operators.store import ExecutionStore
    # dataset id attribute name varies across builds; _doc.id is the usual one.
    _did = getattr(getattr(dataset, "_doc", None), "id", None) or getattr(dataset, "_id", None)
    store = ExecutionStore.create("nureasoning_panel", dataset_id=_did)
for token, summ in token_to_summary.items():
    store.set(token, summ)
print(f"Stored keyframe summaries for {len(token_to_summary)} clip(s) in the execution store "
      f"(no sample-schema changes).")


In [ ]:
from pathlib import Path
import os
import fiftyone as fo

# 2) Write the plugin into FiftyOne's configured plugins directory so the App discovers it on any
#    machine. If no plugins_dir is configured yet, default to ~/fiftyone/__plugins__ and set it for
#    this session. To make it permanent across sessions, run once in a terminal:
#        fiftyone config plugins_dir ~/fiftyone/__plugins__
plugins_dir = fo.config.plugins_dir
if not plugins_dir:
    plugins_dir = str(Path.home() / "fiftyone" / "__plugins__")
    fo.config.plugins_dir = plugins_dir
PLUGINS_DIR = Path(plugins_dir)
PANEL_DIR = PLUGINS_DIR / "nureasoning-reasoning-panel"
PANEL_DIR.mkdir(parents=True, exist_ok=True)

(PANEL_DIR / "fiftyone.yml").write_text(
    "name: nureasoning-reasoning-panel\n"
    "type: plugin\n"
    "author: nuReasoning demo\n"
    "version: 1.0.0\n"
    "description: Synced reasoning panel for the nuReasoning long-tail demo\n"
    "fiftyone:\n"
    '  version: "*"\n'
    "panels:\n"
    "  - nureasoning_reasoning_panel\n"
)

PANEL_CODE = r"""
import os
import json
import base64
import mimetypes
import fiftyone.operators as foo
import fiftyone.operators.types as types


def _img_data_uri(path, max_w=520):
    # Read a local image and return a base64 data URI the panel markdown can render inline.
    # Downsamples to max_w if Pillow is available (keeps the data URI small); else raw bytes.
    if not path or not os.path.exists(path):
        return None
    try:
        from PIL import Image
        import io
        im = Image.open(path).convert("RGB")
        if im.width > max_w:
            im = im.resize((max_w, max(1, round(im.height * max_w / im.width))))
        buf = io.BytesIO()
        im.save(buf, format="JPEG", quality=80)
        data = buf.getvalue()
        mime = "image/jpeg"
    except Exception:
        with open(path, "rb") as f:
            data = f.read()
        mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    return "data:%s;base64,%s" % (mime, base64.b64encode(data).decode("ascii"))


class ReasoningPanel(foo.Panel):
    @property
    def config(self):
        return foo.PanelConfig(
            name="nureasoning_reasoning_panel",
            label="nuReasoning: Reasoning",
            # show beside the video in the expanded modal AND in the grid
            surfaces="grid modal",
        )

    def on_load(self, ctx):
        ctx.panel.state.kf_index = 0
        self._load_current(ctx)

    def on_change_current_sample(self, ctx):
        # new clip opened in the modal -> reset to its first keyframe
        ctx.panel.state.kf_index = 0
        self._load_current(ctx)

    def _summary(self, ctx):
        # Resolve the current clip's token from the active sample, then read its keyframe
        # summary from the dataset execution store (NOT from a sample field).
        sample = getattr(ctx, "current_sample", None)
        if isinstance(sample, str):
            try:
                sample = ctx.dataset[sample]
            except Exception:
                sample = None
        token = None
        if sample is not None:
            try:
                token = sample["clip_token"]
            except Exception:
                token = None
        if token is None:
            return []
        try:
            store = ctx.store("nureasoning_panel")
            data = store.get(token)
        except Exception:
            data = None
        if not data:
            return []
        # stored as a list already; tolerate a JSON string just in case
        if isinstance(data, str):
            try:
                data = json.loads(data)
            except Exception:
                return []
        return data if isinstance(data, list) else []

    def _load_current(self, ctx):
        kfs = self._summary(ctx)
        ctx.panel.state.n_kf = len(kfs)

    def set_kf(self, ctx):
        # called by the prev/next buttons; params carry the delta
        delta = ctx.params.get("delta", 0)
        n = ctx.panel.state.get("n_kf", 0) or 0
        i = (ctx.panel.state.get("kf_index", 0) or 0) + delta
        if n:
            i = max(0, min(i, n - 1))
        ctx.panel.state.kf_index = i

    def render(self, ctx):
        panel = types.Object()
        kfs = self._summary(ctx)

        if not kfs:
            panel.md(
                "### nuReasoning\n_Open a clip in the modal to see its labeled keyframes._",
                name="empty",
            )
            return types.Property(
                panel, view=types.GridView(align_x="center", orientation="vertical")
            )

        i = ctx.panel.state.get("kf_index", 0) or 0
        i = max(0, min(i, len(kfs) - 1))
        kf = kfs[i]

        # keyframe navigation
        btns = panel.menu("nav", variant="contained")
        btns.btn("prev", label="◀ Prev", on_click=self.set_kf, params={"delta": -1})
        btns.btn("next", label="Next ▶", on_click=self.set_kf, params={"delta": 1})

        dec = kf.get("decision", {}) or {}
        lon = dec.get("Longitudinal") or "n/a"
        lat = dec.get("Lateral") or "n/a"

        md = []
        # Compact position line.
        md.append(
            f"**Keyframe {i + 1} of {len(kfs)}**  ·  frame {kf.get('frame')}  ·  t = {kf.get('t')}s"
        )
        # Keyframe image (front camera), embedded inline so the panel shows what the car sees.
        uri = _img_data_uri(kf.get("image"))
        if uri:
            md.append(f"![keyframe]({uri})")
        # Decision as a normal-size bold headline (no oversized header, no emoji).
        md.append(f"**Decision:** {lon}  /  {lat}")

        # Scene
        scene = kf.get("scene")
        if scene:
            md.append(f"**Scene**  \n{scene}")

        # Critical components as a tight list
        crit = kf.get("critical") or {}
        if crit:
            lines = ["**Critical components**"]
            for name, attrs in crit.items():
                attrs = attrs or {}
                if attrs:
                    bits = ", ".join(f"{k} {v}" for k, v in attrs.items())
                    lines.append(f"- **{name}** — {bits}")
                else:
                    lines.append(f"- **{name}**")
            md.append("\n".join(lines))

        # Why / reasoning trace
        if kf.get("trace"):
            md.append(f"**Why**  \n{kf['trace']}")

        # Counterfactual: parse the stored text into a readable list with bolded risk levels,
        # instead of dumping it in a monospace code block.
        cf = kf.get("counterfactual")
        if cf:
            cf_lines = ["**Counterfactual — options weighed**"]
            for raw in str(cf).splitlines():
                s = raw.strip()
                if not s:
                    continue
                if s.endswith(":"):                       # section header line
                    cf_lines.append(f"\n_{s[:-1]}_")
                    continue
                # lines look like "[Risk] move — reason"; bold the risk tag
                if s.startswith("[") and "]" in s:
                    risk, rest = s[1:].split("]", 1)
                    badge = {"Unsafe": "🔴", "Suboptimal": "🟡", "Safe": "🟢"}.get(risk.strip(), "•")
                    cf_lines.append(f"- {badge} **{risk.strip()}** —{rest}")
                else:
                    cf_lines.append(f"- {s}")
            md.append("\n".join(cf_lines))

        panel.md("\n\n".join(md), name="body")

        return types.Property(
            panel, view=types.GridView(align_x="left", orientation="vertical")
        )


def register(p):
    p.register(ReasoningPanel)
"""
(PANEL_DIR / "__init__.py").write_text(PANEL_CODE)

# Point FiftyOne at this plugins dir (process-local; persists if you also export it in your shell).
import fiftyone as fo
existing = fo.config.plugins_dir
if not existing or str(PLUGINS_DIR) not in str(existing):
    fo.config.plugins_dir = str(PLUGINS_DIR)
os.environ["FIFTYONE_PLUGINS_DIR"] = str(PLUGINS_DIR)

print("Plugin written to:", PANEL_DIR)
print("plugins_dir       :", fo.config.plugins_dir)
print("Files             :", [p.name for p in PANEL_DIR.iterdir()])


In [ ]:
# 3) Refresh so the App discovers the new plugin, then open the panel in the modal.
#    If the panel doesn't appear in the App, confirm it's listed and enabled with:
#        !fiftyone plugins list
#    and that the plugins directory it reports matches PLUGINS_DIR printed above.
session.refresh()

# Open it programmatically next to the video in the modal. You can also add it manually via the
# "+" next to the Samples tab -> "nuReasoning: Reasoning".
try:
    session.open_panel("nureasoning_reasoning_panel", is_active=True)
except Exception as e:
    print("Open it manually via the '+' tab. (programmatic open said:", e, ")")

print("Reasoning panel ready. Open a clip in the modal; use Prev/Next keyframe to step through.")


## Step 9 — Cleanup (optional)

The dataset was created with `persistent=True` so it survives kernel restarts during rehearsal.
Remove it (and the scratch media) when you're done.

In [ ]:
# import shutil
# fo.delete_dataset(DATASET_NAME)
# shutil.rmtree(DEMO_BUILD_DIR, ignore_errors=True)
# print("Removed dataset and build dir.")
